# DC-Ops on Alif E8: overnight training (Colab)

Runs the whole distillation pipeline on this Colab GPU:

1. **Teacher**: the DC-Ops YOLOv8n-seg labels component crops; a ConvNeXt teacher is fine-tuned on them and writes soft labels.
2. **Students** (6 runs): MobileNetV2 for the Ethos-U85, distilled from the teacher; 128/160/192 px, a long run, and two ablations (no distillation, no board-camera augmentation).
3. **Voice** (2 runs): DS-CNN keyword spotters distilled from the AST speech-commands model, on the board's exact MFCC.

**Before you start**
- *Runtime > Change runtime type*: pick a GPU (A100 or L4 if you have them, else T4).
- Add your Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left bar) and enable notebook access.
- Keep this tab open overnight (and the Mac awake). If it disconnects, just *Run all* again: finished jobs are skipped.

Results go to Google Drive (`MyDrive/dcops_e8/`) and to the private Hub repo `abhijitbetigeri/dcops-e8-student`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
try:  # Drive keeps results across disconnects; without it, everything still goes to the Hub
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/dcops_e8'
except Exception as e:
    print('Drive not mounted, using local disk:', e)
    OUT = '/content/dcops_e8'
os.makedirs(f'{OUT}/logs', exist_ok=True)
print('outputs ->', OUT)

In [ ]:
!pip -q install timm ultralytics transformers soundfile huggingface_hub
from huggingface_hub import hf_hub_download
import shutil
os.makedirs('/content/jobs', exist_ok=True)
for f in ['teacher.py', 'student.py', 'kws.py']:
    shutil.copy(hf_hub_download('abhijitbetigeri/dcops-e8-jobs', f, force_download=True), f'/content/jobs/{f}')
print(os.listdir('/content/jobs'))

In [ ]:
# Scheduler: runs jobs in parallel on the one GPU, as many at once as its memory allows.
# Resumable: a job is skipped when its result is already on Drive.
import subprocess, time, torch, glob

vram = torch.cuda.get_device_properties(0).total_memory / 2**30
SLOTS = 6 if vram > 60 else 4 if vram > 35 else 3 if vram > 20 else 2
print(f'GPU {torch.cuda.get_device_name(0)}, {vram:.0f} GB -> {SLOTS} jobs at a time')

CROPS = f'{OUT}/teacher/crops.npz'
JOBS = [  # name, command, result that marks it done, needs teacher crops
  ('teacher',      'teacher.py --no-push',                                         f'{OUT}/teacher/meta.json', False),
  ('kws_s_kd',     'kws.py --name dscnn_s_kd --ch 64 --kd-alpha 0.5 --data /content/sc_s',  f'{OUT}/kws_dscnn_s_kd/metrics.json', False),
  ('kws_m_kd',     'kws.py --name dscnn_m_kd --ch 128 --kd-alpha 0.5 --data /content/sc_m', f'{OUT}/kws_dscnn_m_kd/metrics.json', False),
  ('r160_kd',      f'student.py --name r160_kd --res 160 --kd-alpha 0.7 --crops {CROPS}',          f'{OUT}/run_r160_kd/metrics.json', True),
  ('r128_kd',      f'student.py --name r128_kd --res 128 --kd-alpha 0.7 --crops {CROPS}',          f'{OUT}/run_r128_kd/metrics.json', True),
  ('r192_kd',      f'student.py --name r192_kd --res 192 --kd-alpha 0.7 --crops {CROPS}',          f'{OUT}/run_r192_kd/metrics.json', True),
  ('r160_nokd',    f'student.py --name r160_nokd --res 160 --kd-alpha 0 --crops {CROPS}',          f'{OUT}/run_r160_nokd/metrics.json', True),
  ('r160_noaug',   f'student.py --name r160_noaug --res 160 --kd-alpha 0.7 --domain-aug 0 --crops {CROPS}', f'{OUT}/run_r160_noaug/metrics.json', True),
  ('r160_kd_e120', f'student.py --name r160_kd_e120 --res 160 --kd-alpha 0.7 --epochs 120 --crops {CROPS}', f'{OUT}/run_r160_kd_e120/metrics.json', True),
]

def after(name):
    # copy results to Drive (the scripts write to /tmp); students/kws also push to the Hub
    if name == 'teacher':
        shutil.copytree('/tmp/out', f'{OUT}/teacher', dirs_exist_ok=True)
        from huggingface_hub import HfApi
        api = HfApi(); api.create_repo('abhijitbetigeri/dcops-e8-crops', repo_type='dataset', private=True, exist_ok=True)
        for f in ('crops.npz', 'teacher.pt', 'meta.json'):
            api.upload_file(path_or_fileobj=f'/tmp/out/{f}', path_in_repo=f, repo_id='abhijitbetigeri/dcops-e8-crops', repo_type='dataset')
    elif name.startswith('kws'):
        run = {'kws_s_kd': 'dscnn_s_kd', 'kws_m_kd': 'dscnn_m_kd'}[name]  # the --name the job was given
        shutil.copytree(f'/tmp/kws_{run}', f'{OUT}/kws_{run}', dirs_exist_ok=True)
    else:
        shutil.copytree(f'/tmp/run_{name}', f'{OUT}/run_{name}', dirs_exist_ok=True)

pending = [j for j in JOBS if not os.path.exists(j[2])]
print('to run:', [j[0] for j in pending])
running = {}
t0 = time.time()
while pending or running:
    teacher_done = os.path.exists(CROPS) and os.path.exists(f'{OUT}/teacher/meta.json')
    for j in list(pending):
        if len(running) >= SLOTS: break
        if j[3] and not teacher_done: continue
        log = open(f"{OUT}/logs/{j[0]}.log", 'a')
        running[j[0]] = (subprocess.Popen(f'python /content/jobs/{j[1]}', shell=True, stdout=log, stderr=subprocess.STDOUT), j)
        pending.remove(j); print(time.strftime('%H:%M'), 'started', j[0])
    for name, (p, j) in list(running.items()):
        if p.poll() is not None:
            del running[name]
            if p.returncode == 0:
                try: after(name); print(time.strftime('%H:%M'), 'done', name)
                except Exception as e: print('copy failed', name, e)
            else:
                print(time.strftime('%H:%M'), f'FAILED {name} (exit {p.returncode}); tail of log:')
                print(open(f'{OUT}/logs/{name}.log').read()[-1500:])
    time.sleep(60)
    if int(time.time() - t0) % 1800 < 60:
        print(time.strftime('%H:%M'), 'running:', list(running), 'pending:', [j[0] for j in pending])
print('ALL DONE in', round((time.time() - t0) / 3600, 1), 'h')

In [ ]:
# Summary
import json
for f in sorted(glob.glob(f'{OUT}/run_*/metrics.json')):
    m = json.load(open(f)); print(f"{m['name']:14s} val {m['val_acc']:.3f}  board-sim {m['val_acc_board_sim']:.3f}")
for f in sorted(glob.glob(f'{OUT}/kws_*/metrics.json')):
    m = json.load(open(f)); print(f"{m['name']:14s} test {m['test_acc']:.4f}  params {m['params']}")